In [1]:
from langchain_community.document_loaders import PyPDFLoader

file_path = r"C:\Users\Técnico em IA\Documents\Arthur Steckelberg Técnico IA\UC15 - NLP\senac_ia_uc15_nlp_projeto\senac_ia_uc15_nlp_project\data\marcelo\sindilojas_2025_2026.pdf"
loader = PyPDFLoader(file_path)
docs = loader.load()

print("Pages loaded:", len(docs))

Pages loaded: 38


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      # you can adjust
    chunk_overlap=100,   # you can adjust
    add_start_index=True
)
all_splits = text_splitter.split_documents(docs)
print("Chunks:", len(all_splits))

Chunks: 299


In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Multilingual sentence embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
)

vector_store = Chroma(
    collection_name="arthur_notebook3_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db_notebook3",  # separate dir from notebook2
)

ids = vector_store.add_documents(all_splits)
print("Stored vectors:", len(ids))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Stored vectors: 299


In [4]:
def retrieve_candidates(query: str, k: int = 10):
    """
    Bi-encoder retrieval from Chroma.
    Returns a list of (Document, retriever_score).
    """
    return vector_store.similarity_search_with_score(query, k=k)

In [5]:
from sentence_transformers import CrossEncoder

# Multilingual cross-encoder for reranking
# If this one errors, fall back to "cross-encoder/ms-marco-MiniLM-L-6-v2"
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MultiBERT-L-12")

OSError: cross-encoder/ms-marco-MultiBERT-L-12 is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

In [ ]:
def rerank_with_cross_encoder(query: str, candidates, top_n: int = 3):
    """
    candidates: list of (Document, retriever_score)
    returns: list of (Document, retriever_score, cross_score) sorted by cross_score desc
    """
    if not candidates:
        return []

    pairs = [(query, doc.page_content) for doc, _ in candidates]
    cross_scores = cross_encoder.predict(pairs)

    reranked = []
    for (doc, ret_score), ce_score in zip(candidates, cross_scores):
        reranked.append((doc, float(ret_score), float(ce_score)))

    reranked.sort(key=lambda x: x[2], reverse=True)
    return reranked[:top_n]

In [ ]:
def answer_query(query: str, k: int = 10, top_n: int = 3):
    candidates = retrieve_candidates(query, k=k)
    reranked = rerank_with_cross_encoder(query, candidates, top_n=top_n)

    for i, (doc, ret_score, ce_score) in enumerate(reranked, start=1):
        print(f"Rank {i}")
        print(f"  retriever_score = {ret_score:.4f}")
        print(f"  cross_score     = {ce_score:.4f}\n")
        print(doc.page_content[:800])  # snippet
        print("\n" + "-" * 80 + "\n")

    return reranked

In [ ]:
answer_query(
    "Quais obrigações a empresa deve cumprir, em caso de parcelamento da rescisão?",
    k=10,
    top_n=3,
)